# ⚡ Direct Phone DroidCam IP Stream (Ultra-Low Latency)
### Important Pre-requisites:
1. **Completely close the DroidCam PC client on Windows** (the DroidCam mobile app only accepts a single client connection at a time).
2. Open the **DroidCam** mobile app; note the **IP and Port shown on the screen** and enter them in Step 1 (or leave blank for auto-discovery).
3. Ensure your phone and PC are connected to the same Wi-Fi / hotspot (with any VPN disabled). If using a **USB connection** with `adb forward tcp:4747 tcp:4747`, set the IP to `127.0.0.1`.

## Step 1: Configuration (Phone IP & Model Settings)

In [7]:
import os
os.environ.setdefault("MPLBACKEND", "Agg")

# ================= Main Configuration =================
PHONE_IP = "10.52.210.200"               # IP displayed in the DroidCam mobile app (e.g., "192.168.43.1"); empty = auto-detect
DROIDCAM_PORT = 4747        # Default DroidCam port on mobile device

TASK_MODE = "segmentation"  # "segmentation" or "detection"
IMGSZ = 480                 # 480 for maximum FPS and minimal latency (or 640 for higher accuracy)
CONF_THRESHOLD = 0.35       # Minimum detection confidence threshold
ENABLE_LOW_LIGHT = False    # Initially disabled for maximum throughput (toggle live with 'E' key)
DRAW_CONTOURS = False       # Set False for higher FPS
ALPHA = 0.40                # Mask transparency factor

print(f"✅ Configuration set: Mode={TASK_MODE} | imgsz={IMGSZ}")

✅ Configuration set: Mode=segmentation | imgsz=480


## Step 2: Auto-Discovery & Direct Connection to DroidCam Stream
This cell directly extracts JPEG byte streams from the HTTP socket instead of relying on OpenCV's buffered `cv2.VideoCapture`. This drops stale frames and reduces latency to near zero.

In [9]:
import re
import time
import socket
import threading
import subprocess
import urllib.request
import cv2
import numpy as np
import torch
from ultralytics import YOLO
from IPython.display import display, Image

def find_droidcam_stream_url(manual_ip="", port=4747):
    candidates = []
    if manual_ip.strip():
        candidates.append(manual_ip.strip())
    try:
        out = subprocess.check_output("ipconfig", shell=True).decode(errors="ignore")
        gateways = re.findall(r"Default Gateway[^\d]*(\d+\.\d+\.\d+\.\d+)", out)
        candidates.extend(gateways)
    except Exception:
        pass
    candidates.extend(["127.0.0.1", "192.168.43.1", "192.168.137.1"])
    candidates = list(dict.fromkeys([ip for ip in candidates if ip]))

    print(f"🔍 Scanning for DroidCam port {port} on candidates: {candidates}")
    for ip in candidates:
        try:
            s = socket.create_connection((ip, port), timeout=1.5)
            s.close()
            for path in ["/mjpegfeed?640x480", "/video"]:
                url = f"http://{ip}:{port}{path}"
                try:
                    req = urllib.request.urlopen(url, timeout=2.5)
                    chunk = req.read(1024)
                    req.close()
                    if len(chunk) > 0:
                        print(f"✅ Direct DroidCam stream discovered: {url}")
                        return url
                except Exception:
                    continue
        except Exception:
            print(f"❌ {ip}:{port} is unreachable.")
    return None

class DirectDroidCamIPStream:
    """Direct MJPEG HTTP stream reader that always drops stale buffer frames for real-time zero-latency."""
    def __init__(self, url):
        self.url = url
        self.running = False
        self.frame = None
        self.frame_id = 0
        self.lock = threading.Lock()

    def start(self):
        if not self.url:
            raise ValueError("❌ Stream URL not found. Ensure DroidCam PC app is closed and PHONE_IP is correctly configured.")
        self.running = True
        self.thread = threading.Thread(target=self._reader_loop, daemon=True)
        self.thread.start()
        t_end = time.time() + 5.0
        while self.frame is None and time.time() < t_end:
            time.sleep(0.02)
        if self.frame is None:
            self.running = False
            raise RuntimeError(f"❌ Connected to {self.url}, but received no video frames.")
        return self

    def _reader_loop(self):
        try:
            stream = urllib.request.urlopen(self.url, timeout=5)
            buf = b""
            while self.running:
                chunk = stream.read(16384)
                if not chunk:
                    break
                buf += chunk
                # Always parse the latest full JPEG frame and discard older backlog
                end = buf.rfind(b"\xff\xd9")
                if end != -1:
                    start = buf.rfind(b"\xff\xd8", 0, end)
                    if start != -1:
                        jpg = buf[start:end + 2]
                        buf = buf[end + 2:]
                        decoded = cv2.imdecode(np.frombuffer(jpg, dtype=np.uint8), cv2.IMREAD_COLOR)
                        if decoded is not None:
                            with self.lock:
                                self.frame = decoded
                                self.frame_id += 1
                elif len(buf) > 250000:
                    last_start = buf.rfind(b"\xff\xd8")
                    buf = buf[last_start:] if last_start != -1 else b""
        except Exception as e:
            print(f"⚠️ Error reading stream: {e}")
            self.running = False

    def read(self):
        with self.lock:
            if self.frame is None:
                return False, None, 0
            return True, self.frame.copy(), self.frame_id
    
    def stop(self):
        self.running = False
        if hasattr(self, "thread"):
            self.thread.join(timeout=1.0)

_CLAHE = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
def enhance_low_light(frame):
    lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    return cv2.cvtColor(cv2.merge((_CLAHE.apply(l), a, b)), cv2.COLOR_LAB2BGR)

STREAM_URL = find_droidcam_stream_url(PHONE_IP, DROIDCAM_PORT)
if STREAM_URL:
    test_s = DirectDroidCamIPStream(STREAM_URL).start()
    ok, f_test, _ = test_s.read()
    test_s.stop()
    if ok and f_test is not None:
        print(f"📸 Frame test succeeded! Resolution: {f_test.shape[1]}x{f_test.shape[0]}")
        _, enc = cv2.imencode(".jpg", f_test)
        display(Image(data=enc.tobytes()))

🔍 Scanning for DroidCam port 4747 on candidates: ['10.52.210.200', '127.0.0.1', '192.168.43.1', '192.168.137.1']
✅ Direct DroidCam stream discovered: http://10.52.210.200:4747/video


RuntimeError: ❌ Connected to http://10.52.210.200:4747/video, but received no video frames.

## Step 3: Model Loading & Inference Warm-up

In [10]:
use_gpu = torch.cuda.is_available()
device = 0 if use_gpu else "cpu"
use_half = use_gpu

if TASK_MODE == "segmentation":
    candidates = ["best.onnx", "best.pt", "yolo11n-seg.pt", "yolo11s-seg.pt"]
else:
    candidates = ["best_det.onnx", "best_det.pt", "yolo11n.pt", "yolo11s.pt"]

selected_model = next((m for m in candidates if os.path.exists(m)), candidates[-2])
print(f"🧠 Loading model: {selected_model} | Hardware: {'GPU (' + torch.cuda.get_device_name(0) + ')' if use_gpu else 'CPU'}")
model = YOLO(selected_model)
half_flag = use_half and not selected_model.endswith(".onnx")

# Warm-up run
_ = model.predict(np.zeros((IMGSZ, IMGSZ, 3), dtype=np.uint8), imgsz=IMGSZ, device=device, half=half_flag, verbose=False)
print("✅ Model warm-up completed and ready for inference!")

🧠 Loading model: best.pt | Hardware: GPU (NVIDIA GeForce RTX 3050 Laptop GPU)
✅ Model warm-up completed and ready for inference!


## Step 4: Real-Time Stream Inference (`Q`: Quit | `E`: Toggle Low-Light Filter)

In [13]:
CLASS_COLORS = {
    0: (0, 165, 255),   # exposed_body
    1: (255, 80, 50),   # clothing
    2: (0, 255, 0),     # person
    3: (140, 140, 140)  # debris
}

stream = DirectDroidCamIPStream(STREAM_URL).start()
prev_time = time.time()
fps = 0.0
enh_state = ENABLE_LOW_LIGHT
last_fid = -1

print("🚀 Direct low-latency DroidCam IP stream active!")

try:
    while True:
        ok, frame, fid = stream.read()
        if not ok or frame is None or fid == last_fid:
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break
            time.sleep(0.002)
            continue
        last_fid = fid

        t0 = time.time()
        proc_frame = enhance_low_light(frame) if enh_state else frame

        results = model.predict(
            proc_frame,
            conf=CONF_THRESHOLD,
            imgsz=IMGSZ,
            device=device,
            half=half_flag,
            retina_masks=False,
            verbose=False
        )

        if results and len(results) > 0:
            res = results[0]
            h, w = proc_frame.shape[:2]

            if TASK_MODE == "segmentation" and res.masks is not None:
                overlay = proc_frame.copy()
                masks = res.masks.data.cpu().numpy()
                classes = res.boxes.cls.cpu().numpy().astype(int)
                for mask, cid in zip(masks, classes):
                    col = CLASS_COLORS.get(cid, (0, 255, 0)) if len(model.names) <= 4 else ((0, 255, 0) if cid == 0 else (180, 180, 180))
                    m_bin = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST) > 0.5
                    overlay[m_bin] = col
                    if DRAW_CONTOURS:
                        cnts, _ = cv2.findContours(m_bin.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                        cv2.drawContours(proc_frame, cnts, -1, col, 2)
                cv2.addWeighted(overlay, ALPHA, proc_frame, 1.0 - ALPHA, 0, proc_frame)

            if res.boxes is not None:
                boxes = res.boxes.xyxy.cpu().numpy().astype(int)
                classes = res.boxes.cls.cpu().numpy().astype(int)
                confs = res.boxes.conf.cpu().numpy()
                for (x1, y1, x2, y2), cid, score in zip(boxes, classes, confs):
                    col = CLASS_COLORS.get(cid, (0, 255, 0)) if len(model.names) <= 4 else ((0, 255, 0) if cid == 0 else (200, 200, 200))
                    name = model.names.get(cid, str(cid))
                    cv2.rectangle(proc_frame, (x1, y1), (x2, y2), col, 2)
                    cv2.putText(proc_frame, f"{name} {score*100:.0f}%", (x1, max(16, y1 - 6)),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, col, 2)

        now = time.time()
        fps = 0.9 * fps + 0.1 * (1.0 / (now - prev_time + 1e-6))
        prev_time = now
        lat_ms = (now - t0) * 1000

        cv2.rectangle(proc_frame, (8, 8), (385, 62), (20, 20, 20), -1)
        cv2.putText(proc_frame, f"DroidCam IP | FPS: {fps:.1f} ({lat_ms:.0f}ms)", (16, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 255), 2)
        cv2.putText(proc_frame, f"CLAHE (E): {'ON' if enh_state else 'OFF'} | Quit: 'Q'", (16, 52),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (200, 255, 200), 1)

        cv2.imshow("DroidCam IP -> USAR Low-Latency", proc_frame)
        key = cv2.waitKey(1) & 0xFF
        if key == ord("q"):
            break
        elif key == ord("e"):
            enh_state = not enh_state

except KeyboardInterrupt:
    print("🛑 Interrupted by user.")
finally:
    stream.stop()
    cv2.destroyAllWindows()
    print("✅ Stream stopped and display windows closed.")

🚀 Direct low-latency DroidCam IP stream active!
✅ Stream stopped and display windows closed.
